# Serving, without a GPU

Toy backend only. Micro-batching two retrieve calls must match two sequential calls.
Unconstrained beam search can emit semantic IDs that are not in the catalog; the trie path cannot.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from backends.toy import ToyBackend
from batching import Batcher

backend = ToyBackend(n_items=16)
reqs = [{"history_items": [1, 2], "k": 5}, {"history_items": [4], "k": 5}]
batched = Batcher(backend.retrieve_many).flush(reqs)
print("batched", [row["items"] for row in batched])
print("solo   ", [backend.retrieve_many([r])[0]["items"] for r in reqs])

In [ ]:
constrained = backend.generate_many([{"history_items": [1], "beam_width": 8, "constrained": True}])[0]
free = backend.generate_many([{"history_items": [1], "beam_width": 8, "constrained": False}])[0]
print("constrained invalid", constrained["invalid_rate"], "items", constrained["items"][:5])
print("unconstrained invalid", free["invalid_rate"])